# Milestone 1 — Data Crawling: Implementation, Validation and Statistics

This notebook is the first section of the report. It follows the order of the project's specification: **entry discovery → full-text source download → reference extraction → data organization → report statistics → scraper performance**.

It combines three earlier notebooks:

| Part | Sections | Content | 
| :-- | :-- | :-- | 
| I. Implementation | 1 – 7 | How the two crawlers work: libraries, API calls, data flow, what is scraped, output layout | 
| II. Validation | 8 | Success rate and version accuracy against the Kaggle ground truth | 
| III. Statistics and performance | 9 – 11 | Paper sizes before/after removing figures, references, running time, RAM and disk | 
| IV. Conclusions | 12 | Summary generated from the computed numbers | 

All tables and figures from Part II onward are **generated when the notebook is executed** from the crawler output folders; no number is typed in by hand.

In [ ]:
import json, re
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.rcParams.update({
    "figure.autolayout": True, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 120)

# ---------------------------------------------------------------- configuration
# Assigned range: must match the constants at the top of src/arxiv_crawler.py.
YEAR_MONTH, START_SEQ, END_SEQ = "2510", 10788, 10837
EXPECTED_IDS = [f"{YEAR_MONTH}.{n:05d}" for n in range(START_SEQ, END_SEQ + 1)]

# The notebook looks for its inputs in these folders (run it from the repo root or from src/).
SEARCH_DIRS = [Path("."), Path("src"), Path(".."), Path("../src"), Path("../../arxiv")]

def find_path(name, directory=False):
    for base in SEARCH_DIRS:
        p = base / name
        if directory and p.is_dir() and any(p.iterdir()):
            return p
        if not directory and p.is_file():
            return p
    return None

DIR_ARXIV = find_path("sourcesArxiv", directory=True)   # output of arxiv_crawler.py
DIR_OAI   = find_path("sourcesOAI", directory=True)     # output of oai_crawler.py
KAGGLE_JSONL = find_path("papers_hoang.jsonl") or find_path("arxiv-metadata-oai-snapshot.json")  # ground truth (JSON lines)
BENCH_RESULTS = find_path("benchmark_results.csv")
BENCH_TIMESERIES = find_path("benchmark_timeseries.csv")

LABELS = {"arxiv": "arXiv API", "oai": "OAI-PMH"}
COLORS = {"arxiv": "#0072B2", "oai": "#E69F00", "kaggle": "#7F7F7F"}
RESULTS = {}   # filled by later cells, used by the final summary

print(f"Assigned range : {EXPECTED_IDS[0]} .. {EXPECTED_IDS[-1]}  ({len(EXPECTED_IDS)} papers)")
for label, p in [("arXiv crawler output", DIR_ARXIV), ("OAI crawler output", DIR_OAI), ("Kaggle ground truth", KAGGLE_JSONL),
                 ("Benchmark results", BENCH_RESULTS), ("Benchmark time series", BENCH_TIMESERIES)]:
    print(f"{label:<22}: {p if p else 'not found'}")

# Part I — Implementation of the crawlers

## 1. System overview

The system consists of **two independent crawlers** that run the same three-stage pipeline. They differ only in how they *discover* papers and learn their version history:

- `src/arxiv_crawler.py` — discovery through the **arXiv API** (Atom/XML, wrapped by the `arxiv` Python client).
- `src/oai_crawler.py` — discovery through the **OAI-PMH** protocol (`arXivRaw` XML requested directly with `requests`).

Pipeline shared by both crawlers:

1. **Discovery**: resolve each assigned arXiv ID to its metadata and to the list of its versions.
2. **Source download**: for every version, request the versioned source archive from `export.arxiv.org/src/` and keep only the `.tex` and `.bib` files.
3. **References**: ask Semantic Scholar once per paper for its reference list and store a copy beside every version.

Data flow:

```text
assigned IDs (2510.10788 ... 2510.10837)
      │
      ▼
 ┌─────────────── 1. DISCOVERY ───────────────┐
 │ arxiv_crawler : 1 API call per 50 IDs      │──► metadata + latest version number (vN)
 │ oai_crawler   : 1 GetRecord call per ID    │──► metadata + full version history (v1..vN)
 └────────────────────────────────────────────┘
      │ for every paper
      ▼
 ┌─────────────── 3. REFERENCES ──────────────┐
 │ GET api.semanticscholar.org/graph/v1/paper/│──► list of cited papers
 │     arXiv:<base_id>   (once per paper)     │
 └────────────────────────────────────────────┘
      │ for every version v1..vN
      ▼
 ┌─────────────── 2. SOURCE DOWNLOAD ─────────┐
 │ write metadata.json                        │
 │ write references.json                      │
 │ GET export.arxiv.org/src/<id>v<N> → bytes  │
 │ open bytes in memory with tarfile          │
 │ keep only *.tex / *.bib → write to disk    │
 │ rewrite metadata.json with final status    │
 └────────────────────────────────────────────┘
      │
      ▼
 sourcesArxiv/<base_id>/v<N>/   or   sourcesOAI/<base_id>/v<N>/
```

Having two crawlers on identical IDs allows the report to compare them on correctness (section 8), size and reference statistics (sections 9–10) and performance (section 11).

## 2. Libraries and tools

| Library / tool | What it does in this project | Used by |
| :-- | :-- | :-- |
| `arxiv` (arxiv.py client) | Builds `arxiv.Search(id_list=...)`, sends the API request, paginates, enforces the 3 s delay, retries, and parses Atom XML into `Result` objects | `arxiv_crawler` |
| `requests` | HTTP GET with `params`, `headers`, `timeout` for the OAI-PMH endpoint and for the Semantic Scholar Graph API | both |
| `urllib.request`, `urllib.error`, `http.client` | Downloads the source archive from `/src/`; `HTTPError` codes and `IncompleteRead` decide whether to retry | both |
| `xml.etree.ElementTree` | Parses OAI-PMH XML (with namespaces) | `oai_crawler` |
| `tarfile`, `gzip`, `io.BytesIO` | Opens the downloaded bytes in memory, iterates archive members, handles single-file gzip submissions | both |
| `pathlib` (`Path`, `PurePosixPath`) | Creates output folders; rejects absolute or `..` paths inside archives | both |
| `json`, `re` | Writes `metadata.json` / `references.json`; strips `vN` suffixes; cleans author strings | both |
| `time` | `sleep` for rate limiting, `monotonic` for the OAI throttle | both |
| `email.utils.parsedate_to_datetime`, `datetime` | Converts OAI RFC-2822 version dates to UTC ISO strings | `oai_crawler` |
| `argparse` | Command-line modes `ids`, `range`, `harvest` | `oai_crawler` |
| `python-dotenv` | Loads the Semantic Scholar `API_KEY` from `.env` | both |
| `time.perf_counter`, `psutil` | Wall-clock timing and RAM sampling in the benchmark (section 11) | benchmark script |
| `pandas`, `numpy`, `matplotlib` | Statistics, tables and figures in this notebook | notebook |

## 3. Stage 1 — Entry discovery

### 3.1 `arxiv_crawler.py` (arXiv API)

**Configuration (Can be changed to match requirements)**

- `YEAR_MONTH = "2510"`, `START_SEQ = 10788`, `END_SEQ = 10837` → 50 IDs generated with `f"{YEAR_MONTH}.{seq:05d}"`.
- `INTERVAL = 3.1` seconds between Semantic Scholar calls.
- `API_KEY` read from `.env`.
- `arxiv.Client(page_size=100, delay_seconds=3.0, num_retries=3)`: the client sleeps 3 s between requests and retries failed pages three times.

**How the API is called**

1. `chunk_list` splits the IDs into batches of 50.
2. For each batch the crawler builds `arxiv.Search(id_list=batch)` and iterates `client.results(search)`.
3. The client sends one request to the arXiv query endpoint (`export.arxiv.org/api/query`) with the `id_list` parameter, receives an **Atom XML** feed, and turns every `<entry>` into a `Result` object.
4. One request therefore resolves up to 50 papers.

**How the version list is found**

- `paper.entry_id` ends with the *latest* version, for example `http://arxiv.org/abs/2510.10788v3`.
- The crawler cuts the suffix (`rsplit("/", 1)[-1]`, then `rpartition("v")`) to get `base_id = 2510.10788` and `latest_version = 3`.
- The API returns only the latest entry, so the versions `v1 … v3` are **inferred** from that number and each is downloaded separately.

**What is scraped from each `Result`** (function `build_paper_metadata`)

- `arxiv_id` (versioned), `version`
- `title` (stripped, newlines removed), `authors` (list of names)
- `submission_date` ← `paper.published` (date of v1)
- `updated_date` ← `paper.updated` (date of the **latest** version)
- `journal_ref`, `primary_category`, `categories`

The same `Result` object is reused for every version of the paper, so the API is not queried again per version.

**Resume logic**

- A paper is skipped when every version `v1…vN` already has a `metadata.json` whose `source_status` is `downloaded` or `no_tex_or_bib`.
- References are re-used from `v1/references.json` when that file exists, otherwise fetched.
- `get_source` also skips a single version whose `metadata.json` is already complete.

### 3.2 `oai_crawler.py` (OAI-PMH)

**Configuration (Can be changed to match requirements)**

- `OAI_ENDPOINT = "https://oaipmh.arxiv.org/oai"` (the endpoint introduced in March 2025).
- `OAI_MIN_DELAY = 3.0` s, `S2_INTERVAL = 3.1` s, `MAX_RETRIES = 8`, `BACKOFF_BASE = 3.0`, `BACKOFF_MAX = 120.0`.
- Custom `User-Agent`, `API_KEY` from `.env`.

**Command-line modes**

- `ids <id> <id> …` — explicit IDs.
- `range <YYMM> <start> <end>` — expands to explicit IDs, as in `arxiv_crawler`.
- `harvest --from … --until … --set … --limit N` — bulk harvesting by date window and OAI set.
- Global options: `--out` (output directory, default `./sourcesOAI`) and `--no-refs` (skip Semantic Scholar).

**How the protocol is called**

- `fetch_oai_record(id)` sends `GET https://oaipmh.arxiv.org/oai` with `verb=GetRecord`, `identifier=oai:arXiv.org:<base_id>`, `metadataPrefix=arXivRaw`. This is **one request per paper**.
- `list_records(...)` sends `verb=ListRecords` with optional `from`, `until`, `set`; it then follows each `<resumptionToken>` (follow-up requests carry only the verb and the token) until the token is empty. A `noRecordsMatch` error ends the harvest quietly.
- `list_identifiers(...)` is implemented the same way for IDs only but is not used by the command-line modes.
- The crawler uses `requests` directly and parses XML with `ElementTree`; it does **not** use Sickle.

**Rate limiting and failures (`oai_request`)**

- `_throttle()` guarantees at least 3 s between any two OAI requests (`time.monotonic`).
- Timeouts and connection errors → exponential backoff `min(3·2^attempt, 120)` s.
- HTTP 503 → wait for the numeric `Retry-After` header when present, otherwise back off.
- HTTP 429 or ≥ 500 → back off and retry (up to 8 attempts).
- Other HTTP errors are raised; an OAI `<error>` element becomes `OAIError`.

**How the XML is turned into metadata (`parse_record`)**

- Deleted records (`header status="deleted"`) and records without an ID are skipped.
- Reads from the `arXivRaw` namespace: `id`, `title`, `authors`, `categories`, `journal-ref`, and one `<version version="vN">` element per version holding a `<date>`.
- `parse_authors` removes parenthesised affiliations, replaces " and " by a comma and splits on commas.
- `_to_iso` converts each RFC-2822 date to a UTC ISO string.
- It returns **one metadata dictionary per version**: `submission_date` is the date of v1 for all versions, `updated_date` is the date of that particular version.

Because the full version history comes with the record, the OAI crawler does not need to infer versions from a "latest" number.

## 4. Stage 2 — Source download and extraction

Both crawlers use the same idea; only the retry details differ.

**Request**

- URL: `https://export.arxiv.org/src/<versioned_id>`, for example `…/src/2510.10788v2`. Each version is fetched on its own.
- Tool: `urllib.request.urlopen` with a descriptive `User-Agent` and a 120 s timeout; the whole response is read into a `bytes` object (no temporary file).
- Pacing: `time.sleep(3)` after every request, including a 404.

**Retry behaviour**

| Situation | `arxiv_crawler` | `oai_crawler` |
| :-- | :-- | :-- |
| HTTP 404 (no source) | treated as empty archive | treated as empty archive |
| `IncompleteRead` | retry (3 attempts, 3 s) | retry (3 attempts, exponential backoff) |
| HTTP 429 / 500 / 502 / 503 / 504 | retry, 5 s wait | retry, `Retry-After` or backoff (also 421) |
| Other HTTP errors | raised | raised |
| Attempts exhausted | continues with empty bytes → status `no_tex_or_bib` | raises `RuntimeError` → status `download_failed` |

**Extraction (in memory, no figures written)**

1. `tarfile.open(fileobj=io.BytesIO(bytes), mode="r:*")` opens the archive with automatic compression detection.
2. For every member the crawler adds `member.size` to `total_uncompressed_bytes` (the size **before** filtering).
3. A member is kept only if it is a regular file, its path is not absolute and contains no `..`, and its suffix is `.tex` or `.bib`.
4. Kept files are written under the version folder with their nested paths preserved; their size is added to `filtered_uncompressed_bytes` (the size **after** filtering).
5. If the bytes are not a tar archive, the crawler tries `gzip.decompress`; when the result contains `\documentclass` or `\begin{document}` it is saved as `main.tex` (single-file submissions).
6. PDF-only or Word submissions produce no `.tex`/`.bib` file, so the version is kept with an empty source and the status `no_tex_or_bib`.

Figure binaries never reach the disk, while `\begin{figure}` and `\includegraphics` lines stay untouched inside the `.tex` files, as required for Milestone 2.

**Statuses written to `metadata.json`**

- `downloading` — written before the request (marks an interrupted run).
- `downloaded` — at least one `.tex` or `.bib` file was extracted.
- `no_tex_or_bib` — the request succeeded but contained no usable source.
- `download_failed` — only `oai_crawler`, after exhausted retries.

## 5. Stage 3 — Reference metadata from Semantic Scholar

Both crawlers contain an almost identical `get_paper_references`.

**Request**

- `requests.get("https://api.semanticscholar.org/graph/v1/paper/arXiv:<base_id>", params=..., headers=..., timeout=30)`.
- The `vN` suffix is removed first, so the call is made **once per paper**, not per version.
- `params["fields"]` = `references.paperId, references.externalIds, references.title, references.authors, references.year` (flat field names, as the API requires).
- `headers = {"x-api-key": API_KEY}`; the function raises a clear error if the key is missing.

**Expected response**

```json
{
  "paperId": "…",
  "references": [
    {"paperId": "…", "externalIds": {"ArXiv": "2301.00001", "DOI": "…", "CorpusId": 123},
     "title": "…", "authors": [{"authorId": "…", "name": "…"}], "year": 2023}
  ]
}
```

**How the response is used**

- `response.json().get("references") or []` is the list stored as `references.json` (all references are kept, with or without an arXiv ID).
- The same list is written into the folder of every version.
- `externalIds.ArXiv` is what later decides whether a reference becomes a row of the published `references` table.

**Rate limiting and errors**

- After a successful call: `time.sleep(3.1)` (more conservative than the documented 1 request/second with a key).
- HTTP 429: wait `3.1 · (attempt + 2)` seconds and retry, up to 10 attempts; if it never succeeds the function returns `None` and the paper is recorded with `references_status = "failed"`.
- `oai_crawler` additionally retries timeouts and connection errors, returns `[]` on HTTP 404 (paper not indexed yet) and catches exceptions at the paper level.
- `arxiv_crawler` calls `raise_for_status()` for every other status, including 404, and its main loop does not catch the exception.

## 6. Data organization

**Folder layout** (one folder per paper, one subfolder per version; `/` in old-style IDs becomes `-`):

```text
sourcesArxiv/                 (or sourcesOAI/)
└── 2510.10788/
    ├── v1/
    │   ├── metadata.json
    │   ├── references.json
    │   ├── main.tex
    │   └── refs.bib
    └── v2/
        └── …
```

**Fields written to `metadata.json`**

| Field | Meaning | Both crawlers |
| :-- | :-- | :-- |
| `arxiv_id`, `version` | Versioned ID and version number | yes |
| `title`, `authors`, `journal_ref` | Paper metadata | yes |
| `primary_category`, `categories` | Subject classes | yes |
| `submission_date` | Date of v1 | yes |
| `updated_date` | arXiv API: date of the latest version (same for all versions). OAI: date of this version | yes |
| `source_status` | `downloaded`, `no_tex_or_bib` or `download_failed` | yes |
| `raw_archive_bytes` | Size of the downloaded archive | yes |
| `total_uncompressed_bytes` | Size of all members before filtering | yes |
| `filtered_uncompressed_bytes` | Size of the kept `.tex`/`.bib` files | yes |
| `references_status`, `references_count` | `success` / `failed` and number of references | yes |
| `tex_files`, `bib_files` | Names of the extracted files | OAI only |

**Mapping to the dataset tables required by the milestone**

| Table column | Source in the crawl output |
| :-- | :-- |
| `papers.arxiv_id`, `title`, `authors` | base ID (folder name), `title`, `authors` |
| `papers.submission_date` | `submission_date` |
| `papers.revised_dates` | `updated_date` of each version (reliable only for the OAI crawler) |
| `papers.venue` | `journal_ref` (nullable) |
| `references.arxiv_id` | citing paper (folder name) |
| `references.ref_arxiv_id` | `externalIds.ArXiv` — only references that have one are kept |
| `references.ref_title`, `ref_authors` | `title`, `[a["name"] for a in authors]` |
| `references.ref_s2_id` | `paperId` |
| `references.ref_submission_date` | not available: only `year` is requested (`publicationDate` is not) |

## 7. Side-by-side comparison and known caveats

| Aspect | `arxiv_crawler.py` | `oai_crawler.py` |
| :-- | :-- | :-- |
| Discovery call | `arxiv.Search(id_list=batch of 50)` | `GetRecord` per ID; `ListRecords` + resumption token for harvests |
| Library for discovery | `arxiv` client (Atom XML) | `requests` + `ElementTree` (OAI-PMH XML) |
| Requests for 50 papers | 1 | 50 (each preceded by a 3 s throttle) |
| Version information | latest version number only → `v1…vN` inferred | full version list with a date per version |
| `updated_date` | latest-version date copied to every version | date of each version |
| Source download | `urllib`, `/src/<id>vN` | `urllib`, `/src/<id>vN` |
| Extraction | `.tex`/`.bib` allow-list; single-gzip fallback | same |
| References | `requests`, Semantic Scholar, once per paper | same, with timeout handling and 404 → `[]` |
| Download failure | recorded as `no_tex_or_bib` | recorded as `download_failed` |
| Resume | paper-level and version-level skip | version-level skip |
| Best suited for | a known list or range of IDs | single records or date/set harvests |

**Caveats of the current code** (they matter when reading the statistics below):

1. `arxiv_crawler` cannot tell a failed download (retries exhausted) from a PDF-only version: both are `no_tex_or_bib`.
2. `arxiv_crawler` does not catch Semantic Scholar errors in its main loop; an HTTP 404 or a network error stops the run (a rerun resumes thanks to the skip logic).
3. The arXiv-API `updated_date` is identical for all versions, so it cannot be used as the list of revised dates.
4. The reference request omits `publicationDate` and `venue`.
5. A paper whose references list is empty is recorded as a *successful* reference crawl, because an empty list is returned for an unindexed paper (`404`) as well as for a paper without references. Section 10 therefore reports both the API success rate and the number of papers with at least one reference.
6. The docstrings of `oai_crawler.py` still mention figure files, but the allow-list contains only `.tex` and `.bib`.

# Part II — Validation of the crawled data

## 8. Success rate and version accuracy

The crawl output of both crawlers is checked against the **Kaggle Cornell arXiv snapshot** (JSON lines, one paper per line, with a `versions` list), which serves as ground truth.

Metrics:

- **Paper success rate** = papers with at least one crawled version ÷ assigned papers.
- **Version accuracy** = papers whose number of crawled versions equals the number of versions in the snapshot ÷ papers covered by the snapshot.
- **Source status per version**: `downloaded` (TeX/Bib found), `no_tex_or_bib` (PDF-only or empty source) or `download_failed`.

### 8.1 Loading the crawl output

In [ ]:
def read_json(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return None

def has_arxiv_id(ref):
    ext = ref.get("externalIds") if isinstance(ref, dict) else None
    return isinstance(ext, dict) and bool(ext.get("ArXiv"))

def collect_crawl_metrics(root, crawler):
    # One row per crawled version, read from metadata.json and references.json.
    rows = []
    root = Path(root) if root else None
    if root is None or not root.exists():
        return pd.DataFrame()
    for paper_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        version_dirs = [d for d in paper_dir.iterdir() if d.is_dir() and re.fullmatch(r"v\d+", d.name)]
        for vdir in sorted(version_dirs, key=lambda d: int(d.name[1:])):
            meta = read_json(vdir / "metadata.json")
            if not isinstance(meta, dict):
                continue
            refs = read_json(vdir / "references.json")
            refs = refs if isinstance(refs, list) else None
            n_refs = len(refs) if refs is not None else int(meta.get("references_count") or 0)
            rows.append({
                "crawler": crawler,
                "paper_id": paper_dir.name,
                "version": int(vdir.name[1:]),
                "source_status": meta.get("source_status", "unknown"),
                "raw_archive_bytes": int(meta.get("raw_archive_bytes") or 0),
                "total_uncompressed_bytes": int(meta.get("total_uncompressed_bytes") or 0),
                "filtered_uncompressed_bytes": int(meta.get("filtered_uncompressed_bytes") or 0),
                "references_status": meta.get("references_status", "unknown"),
                "references_count": n_refs,
                "arxiv_refs_count": sum(has_arxiv_id(r) for r in refs) if refs else 0,
                "primary_category": meta.get("primary_category"),
            })
    return pd.DataFrame(rows)

df_arxiv = collect_crawl_metrics(DIR_ARXIV, "arxiv")
df_oai = collect_crawl_metrics(DIR_OAI, "oai")
CRAWLERS = {k: df for k, df in (("arxiv", df_arxiv), ("oai", df_oai)) if not df.empty}

if not CRAWLERS:
    raise FileNotFoundError("No crawl output found. Run the crawlers first (sourcesArxiv/ and/or sourcesOAI/).")

for k, df in CRAWLERS.items():
    print(f"{LABELS[k]:<10}: {df.paper_id.nunique():>3} papers, {len(df):>3} versions")
display(next(iter(CRAWLERS.values())).head())

### 8.2 Ground truth and comparison table

In [ ]:
def load_ground_truth(path, wanted):
    # Stream the JSON-lines snapshot and keep only the wanted IDs (folder-style: "/" -> "-").
    found = {}
    if not path:
        return found
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue
            key = str(item.get("id", "")).replace("/", "-")
            if key in wanted:
                found[key] = item
                if len(found) == len(wanted):
                    break
    return found

gt = load_ground_truth(KAGGLE_JSONL, set(EXPECTED_IDS))
print(f"Ground truth found for {len(gt)} of {len(EXPECTED_IDS)} assigned papers.")
if not gt:
    print("Version accuracy cannot be computed; point KAGGLE_JSONL to a snapshot that contains the assigned IDs.")

cmp = pd.DataFrame({"arxiv_id": EXPECTED_IDS})
cmp["category"] = cmp.arxiv_id.map(lambda i: (gt[i].get("categories") or "").split()[0] if i in gt and gt[i].get("categories") else None)
cmp["kaggle_versions"] = cmp.arxiv_id.map(lambda i: len(gt[i]["versions"]) if i in gt else np.nan)
for k, df in CRAWLERS.items():
    cmp[f"{k}_versions"] = cmp.arxiv_id.map(df.groupby("paper_id").size()).fillna(0).astype(int)
    cmp[f"{k}_has_tex"] = cmp.arxiv_id.map(df.groupby("paper_id").source_status.apply(lambda s: (s == "downloaded").any())).eq(True)

def verdict(r):
    if pd.isna(r.kaggle_versions):
        return "no ground truth" if any(r[f"{k}_versions"] > 0 for k in CRAWLERS) else "not crawled"
    ok = [r[f"{k}_versions"] == r.kaggle_versions for k in CRAWLERS]
    if all(ok):
        return "match"
    if any(ok):
        return "partial (" + ", ".join(LABELS[k] for k, o in zip(CRAWLERS, ok) if o) + " only)"
    return "mismatch"

cmp["verdict"] = cmp.apply(verdict, axis=1)
cmp.head(10)

### 8.3 Core metrics

In [ ]:
n_expected = len(EXPECTED_IDS)
covered = cmp[cmp.kaggle_versions.notna()]
RESULTS["validation"] = {}
table = OrderedDict()
for k in CRAWLERS:
    ok_papers = int((cmp[f"{k}_versions"] > 0).sum())
    n_versions = int(cmp[f"{k}_versions"].sum())
    correct = int((covered[f"{k}_versions"] == covered.kaggle_versions).sum())
    acc = correct / len(covered) * 100 if len(covered) else np.nan
    RESULTS["validation"][k] = dict(papers=ok_papers, success=ok_papers / n_expected * 100, versions=n_versions,
                                    correct=correct, covered=len(covered), accuracy=acc)
    table[LABELS[k]] = [
        n_expected, ok_papers, f"{ok_papers / n_expected * 100:.1f}%", n_versions,
        len(covered), int(covered.kaggle_versions.sum()) if len(covered) else "n/a",
        correct if len(covered) else "n/a", f"{acc:.1f}%" if len(covered) else "n/a",
    ]
index = ["Assigned papers", "Papers crawled (>= 1 version)", "Paper success rate", "Versions crawled",
         "Papers covered by ground truth", "Ground-truth versions (covered papers)",
         "Papers with correct version count", "Version accuracy"]
display(pd.DataFrame(table, index=index))

### 8.4 Figures

In [ ]:
def bar_labels(ax, bars, fmt="{:.0f}"):
    for b in bars:
        h = b.get_height()
        ax.annotate(fmt.format(h), (b.get_x() + b.get_width() / 2, h), xytext=(0, 3),
                    textcoords="offset points", ha="center", va="bottom", fontsize=9)

# Figure 1: crawled papers / versions / correct version counts vs ground truth.
cats = ["Papers crawled", "Versions crawled", "Papers with correct\nversion count"]
series = []
if len(covered):
    series.append(("Kaggle ground truth (covered papers)", COLORS["kaggle"],
                   [len(covered), int(covered.kaggle_versions.sum()), len(covered)]))
for k in CRAWLERS:
    v = RESULTS["validation"][k]
    series.append((LABELS[k], COLORS[k], [v["papers"], v["versions"], v["correct"] if len(covered) else 0]))

fig, ax = plt.subplots(figsize=(9, 4.8))
x, w = np.arange(len(cats)), 0.8 / len(series)
for i, (name, color, vals) in enumerate(series):
    bars = ax.bar(x + (i - (len(series) - 1) / 2) * w, vals, w, label=name, color=color)
    bar_labels(ax, bars)
ax.set_xticks(x); ax.set_xticklabels(cats)
ax.set_ylabel("Count"); ax.set_title("Crawler output compared with the Kaggle ground truth")
ax.legend(frameon=False)
plt.show()

# Figure 2: source status per crawled version.
status_order = ["downloaded", "no_tex_or_bib", "download_failed"]
status_text = {"downloaded": "TeX/Bib found", "no_tex_or_bib": "No TeX/Bib (PDF-only\nor empty source)", "download_failed": "Download failed"}
fig, ax = plt.subplots(figsize=(9, 4.5))
x, w = np.arange(len(status_order)), 0.8 / len(CRAWLERS)
RESULTS["status"] = {}
for i, (k, df) in enumerate(CRAWLERS.items()):
    counts = df.source_status.value_counts().reindex(status_order).fillna(0).astype(int)
    RESULTS["status"][k] = counts.to_dict()
    bars = ax.bar(x + (i - (len(CRAWLERS) - 1) / 2) * w, counts.values, w, label=LABELS[k], color=COLORS[k])
    bar_labels(ax, bars)
ax.set_xticks(x); ax.set_xticklabels([status_text[s] for s in status_order])
ax.set_ylabel("Number of versions"); ax.set_title("Download status of every crawled version")
ax.legend(frameon=False)
plt.show()

### 8.5 Full audit table

One row per assigned paper: version counts of the ground truth and of both crawlers, whether TeX/Bib files were found, and the verdict.

In [ ]:
cols = ["arxiv_id", "category", "kaggle_versions"] + [c for k in CRAWLERS for c in (f"{k}_versions", f"{k}_has_tex")] + ["verdict"]
audit = cmp[cols].copy()
audit["kaggle_versions"] = audit.kaggle_versions.astype("Int64")
display(audit)
print(audit.verdict.value_counts().to_string())

# Part III — Statistics and performance

## 9. Paper size before and after removing figures

For each crawled version the metadata records:

- **before**: `total_uncompressed_bytes`, the size of every file in the source archive (figures, `.tex`, `.bib`, styles, …);
- **after**: `filtered_uncompressed_bytes`, the size of the `.tex` and `.bib` files that were kept;
- `raw_archive_bytes`: the size of the compressed archive that was downloaded.

Only versions that returned a tar/gzip archive (`raw_archive_bytes > 0`) enter the averages; PDF-only versions have no TeX size to compare. "Average reduction per version" is the mean of the per-version ratios, "total reduction" compares the summed sizes.

In [ ]:
def fmt_bytes(n):
    n = float(n)
    for unit in ("B", "KB", "MB", "GB"):
        if abs(n) < 1024 or unit == "GB":
            return f"{n:,.0f} {unit}" if unit == "B" else f"{n:,.2f} {unit}"
        n /= 1024

def fmt(v, kind):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "n/a"
    return {"bytes": lambda: fmt_bytes(v), "pct": lambda: f"{v:.1f}%", "int": lambda: f"{int(v):,}",
            "sec": lambda: f"{v:,.2f} s", "mb": lambda: f"{v:,.2f} MB"}.get(kind, lambda: f"{v:,.2f}")()

def stats_table(stat_fn, crawlers):
    cols, index = {}, None
    for k, df in crawlers.items():
        s = stat_fn(df)
        index = list(s.keys())
        cols[LABELS[k]] = [fmt(v, kind) for v, kind in s.values()]
    return pd.DataFrame(cols, index=index)

def size_stats(df):
    d = df[df.raw_archive_bytes > 0]
    before, after, raw = d.total_uncompressed_bytes, d.filtered_uncompressed_bytes, d.raw_archive_bytes
    per_version = (1 - after / before.replace(0, np.nan)) * 100
    total_red = (1 - after.sum() / before.sum()) * 100 if before.sum() else np.nan
    return OrderedDict([
        ("Versions crawled", (len(df), "int")),
        ("Versions with a source archive", (len(d), "int")),
        ("Average archive size, downloaded (compressed)", (raw.mean(), "bytes")),
        ("Average size BEFORE removing figures", (before.mean(), "bytes")),
        ("Average size AFTER removing figures (.tex + .bib)", (after.mean(), "bytes")),
        ("Average reduction per version", (per_version.mean(), "pct")),
        ("Total size before", (before.sum(), "bytes")),
        ("Total size after", (after.sum(), "bytes")),
        ("Total reduction", (total_red, "pct")),
    ])

RESULTS["size"] = {}
for k, df in CRAWLERS.items():
    s = size_stats(df)
    RESULTS["size"][k] = {name: v for name, (v, _) in s.items()}
display(stats_table(size_stats, CRAWLERS))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

# (a) mean size before/after per crawler
ax = axes[0]
x, w = np.arange(len(CRAWLERS)), 0.36
before = [RESULTS["size"][k]["Average size BEFORE removing figures"] / 1024 for k in CRAWLERS]
after = [RESULTS["size"][k]["Average size AFTER removing figures (.tex + .bib)"] / 1024 for k in CRAWLERS]
b1 = ax.bar(x - w / 2, before, w, label="Before (all files)", color="#D55E00")
b2 = ax.bar(x + w / 2, after, w, label="After (.tex + .bib)", color="#0072B2")
bar_labels(ax, b1, "{:,.0f}"); bar_labels(ax, b2, "{:,.0f}")
ax.set_xticks(x); ax.set_xticklabels([LABELS[k] for k in CRAWLERS])
ax.set_ylabel("Average size per version (KB)"); ax.set_title("Average size before vs after removing figures")
ax.set_ylim(0, max(before) * 1.25)
ax.legend(frameon=True, facecolor="white", loc="upper center", ncol=2)

# (b) every version: before vs after (log-log)
ax = axes[1]
for k, df in CRAWLERS.items():
    d = df[(df.raw_archive_bytes > 0) & (df.total_uncompressed_bytes > 0) & (df.filtered_uncompressed_bytes > 0)]
    ax.scatter(d.total_uncompressed_bytes / 1024, d.filtered_uncompressed_bytes / 1024, s=22, alpha=0.75,
               label=LABELS[k], color=COLORS[k])
lim = ax.get_xlim()
ax.plot(lim, lim, "--", color="#7F7F7F", lw=1, label="no reduction")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Before (KB)"); ax.set_ylabel("After (KB)")
ax.set_title("Per-version size reduction (below the dashed line = smaller)")
ax.legend(frameon=False)
plt.show()

## 10. References per paper and reference crawl success

Reference lists come from Semantic Scholar once per paper and are copied into every version, so the statistics below count **papers**, not versions.

- **API success rate** = papers whose reference request completed (`references_status == "success"`) ÷ papers.
- **Papers with at least one reference** separates a real success from an empty list, which is also returned when a paper is not yet indexed (caveat 5 in section 7).
- **References with an arXiv ID** are the ones that become rows of the published `references` table.

In [ ]:
def per_paper_refs(df):
    return df.groupby("paper_id").agg(
        refs=("references_count", "max"),
        arxiv_refs=("arxiv_refs_count", "max"),
        ok=("references_status", lambda s: (s == "success").any()),
    )

def reference_stats(df):
    g = per_paper_refs(df)
    n = len(g)
    nonzero = g[g.refs > 0]
    return OrderedDict([
        ("Papers analysed", (n, "int")),
        ("Reference crawl completed (API success)", (int(g.ok.sum()), "int")),
        ("API success rate", (g.ok.mean() * 100, "pct")),
        ("Papers with at least 1 reference", (len(nonzero), "int")),
        ("Share of papers with at least 1 reference", (len(nonzero) / n * 100, "pct")),
        ("Average references per paper (all papers)", (g.refs.mean(), "float")),
        ("Median references per paper", (g.refs.median(), "float")),
        ("Maximum references in one paper", (g.refs.max(), "int")),
        ("Average references per paper (papers with >= 1)", (nonzero.refs.mean() if len(nonzero) else np.nan, "float")),
        ("Average references with an arXiv ID per paper", (g.arxiv_refs.mean(), "float")),
        ("Share of references that carry an arXiv ID", (g.arxiv_refs.sum() / g.refs.sum() * 100 if g.refs.sum() else np.nan, "pct")),
    ])

RESULTS["refs"] = {}
for k, df in CRAWLERS.items():
    RESULTS["refs"][k] = {name: v for name, (v, _) in reference_stats(df).items()}
display(stats_table(reference_stats, CRAWLERS))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

# (a) distribution of references per paper
ax = axes[0]
all_refs = pd.concat([per_paper_refs(df).refs for df in CRAWLERS.values()])
bins = np.linspace(0, max(10, all_refs.max()), 16)
for k, df in CRAWLERS.items():
    r = per_paper_refs(df).refs
    ax.hist(r, bins=bins, alpha=0.55, label=LABELS[k], color=COLORS[k])
first = per_paper_refs(next(iter(CRAWLERS.values()))).refs
ax.axvline(first.mean(), color="#D55E00", ls="--", lw=1.5, label=f"mean {first.mean():.1f}")
ax.axvline(first.median(), color="#009E73", ls=":", lw=1.8, label=f"median {first.median():.1f}")
ax.set_xlabel("References per paper"); ax.set_ylabel("Number of papers")
ax.set_title("Distribution of references per paper"); ax.legend(frameon=False)

# (b) average counts
ax = axes[1]
names = ["Mean refs", "Median refs", "Mean refs\nwith arXiv ID"]
x, w = np.arange(len(names)), 0.8 / len(CRAWLERS)
for i, k in enumerate(CRAWLERS):
    r = RESULTS["refs"][k]
    vals = [r["Average references per paper (all papers)"], r["Median references per paper"], r["Average references with an arXiv ID per paper"]]
    bars = ax.bar(x + (i - (len(CRAWLERS) - 1) / 2) * w, vals, w, label=LABELS[k], color=COLORS[k])
    bar_labels(ax, bars, "{:.1f}")
ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylabel("References per paper")
ax.set_title("Average references per paper"); ax.legend(frameon=False)

# (c) success rates
ax = axes[2]
names = ["API success", "Papers with\n>= 1 reference"]
x, w = np.arange(len(names)), 0.8 / len(CRAWLERS)
for i, k in enumerate(CRAWLERS):
    r = RESULTS["refs"][k]
    vals = [r["API success rate"], r["Share of papers with at least 1 reference"]]
    bars = ax.bar(x + (i - (len(CRAWLERS) - 1) / 2) * w, vals, w, label=LABELS[k], color=COLORS[k])
    bar_labels(ax, bars, "{:.1f}%")
ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylim(0, 110); ax.set_ylabel("% of papers")
ax.set_title("Reference crawl success"); ax.legend(frameon=False)
plt.show()

## 11. Scraper performance

### 11.1 Measurement method

The benchmark script (`benchmark.py`, run end-to-end on the assigned papers with a clean output folder, intendedly on a CPU-only Google Colab instance) records the metrics required by the milestone:

- **Running time**: `time.perf_counter()` for the total wall time, the entry-discovery time, and the processing time of each paper (Semantic Scholar call plus download and extraction of all its versions).
- **Memory footprint**: a background thread samples `psutil.Process(os.getpid()).memory_info().rss` every 0.5 s, giving peak and average RAM.
- **Disk**: maximum disk use during the crawl and the size of the final output folder; compared with the compressed archive size and the uncompressed size from section 9.

The benchmark files are read from `benchmark_results.csv` (one row per crawler) and `benchmark_timeseries.csv` (RAM samples over time). Columns are read by their exact names; `benchmark_timeseries.csv` holds two record types: `ram_sample` (RAM over time) and `paper_time` (time of one paper).

### 11.2 Why the two crawlers are expected to differ

- **Discovery**: `arxiv_crawler` resolves 50 IDs in one API call; `oai_crawler` sends one `GetRecord` per ID with a 3 s throttle, so discovery takes roughly `3 s × number of papers`.
- **Per paper**: both wait 3.1 s for Semantic Scholar and 3 s after each source download, so the cost per paper is dominated by these mandatory pauses and grows with the number of versions.
- **Memory**: archives are processed in memory (`io.BytesIO`), so RAM grows with the largest archive but is released after each version.

In [ ]:
def read_csv(path):
    try:
        return pd.read_csv(path) if path else pd.DataFrame()
    except Exception as exc:
        print(f"Could not read {path}: {exc}")
        return pd.DataFrame()

def crawler_key(value):
    v = str(value).lower()
    return "arxiv" if "arxiv" in v else ("oai" if "oai" in v else v)

bench = read_csv(BENCH_RESULTS)   # one row per crawler
ts = read_csv(BENCH_TIMESERIES)   # record_type = "ram_sample" (RAM over time) or "paper_time" (one row per paper)
if not bench.empty:
    bench["_key"] = bench["crawler"].map(crawler_key)
if not ts.empty:
    ts["_key"] = ts["crawler"].map(crawler_key)

# (label, column of benchmark_results.csv, unit) -- column names exactly as written by the benchmark script
PERF_SPEC = [
    ("Papers crawled", "papers_crawled", "int"),
    ("Versions crawled", "versions_crawled", "int"),
    ("Total wall time (end-to-end)", "total_wall_time_sec", "sec"),
    ("Entry discovery time", "discovery_time_sec", "sec"),
    ("Total paper processing time", "total_paper_process_time_sec", "sec"),
    ("Time per paper: mean", "avg_paper_time_sec", "sec"),
    ("Time per paper: median", "median_paper_time_sec", "sec"),
    ("Time per paper: min", "min_paper_time_sec", "sec"),
    ("Time per paper: max", "max_paper_time_sec", "sec"),
    ("Time per paper: std", "std_paper_time_sec", "sec"),
    ("Baseline RAM", "baseline_ram_mb", "mb"),
    ("Average RAM", "avg_ram_mb", "mb"),
    ("Peak RAM", "peak_ram_mb", "mb"),
    ("RAM overhead (peak - baseline)", "ram_overhead_mb", "mb"),
    ("Downloaded archives (compressed)", "raw_archive_bytes", "bytes"),
    ("Largest single archive", "max_single_archive_bytes", "bytes"),
    ("Size before removing figures", "total_uncompressed_bytes", "bytes"),
    ("Size after removing figures (.tex + .bib)", "filtered_uncompressed_bytes", "bytes"),
    ("Maximum disk usage during the crawl", "max_disk_storage_bytes", "bytes"),
    ("Final output size on disk", "final_output_storage_bytes", "bytes"),
    ("Storage saving vs. uncompressed", "storage_saving_pct", "pct"),
]

def perf_stats(key):
    row = bench[bench["_key"] == key]
    if row.empty:
        return None
    r = row.iloc[0]
    return OrderedDict((label, (float(r[col]), kind)) for label, col, kind in PERF_SPEC
                       if col in bench.columns and pd.notna(r[col]))

if bench.empty:
    print("No benchmark_results.csv found: performance tables are skipped. Run the benchmark script first.")
else:
    missing = [col for _, col, _ in PERF_SPEC if col not in bench.columns]
    if missing:
        print("Columns missing from benchmark_results.csv:", missing)
    perf = {k: perf_stats(k) for k in ("arxiv", "oai") if perf_stats(k)}
    RESULTS["perf"] = {k: {n: v for n, (v, _) in s.items()} for k, s in perf.items()}
    labels = list(dict.fromkeys(n for s in perf.values() for n in s))
    table = pd.DataFrame({LABELS[k]: [fmt(*s[n]) if n in s else "n/a" for n in labels] for k, s in perf.items()}, index=labels)
    display(table)

    # Consistency checks: benchmark run vs the crawl folders analysed in sections 8-10.
    for k, s in perf.items():
        notes = []
        if k in CRAWLERS:
            for label, have in (("Papers crawled", CRAWLERS[k].paper_id.nunique()), ("Versions crawled", len(CRAWLERS[k]))):
                if label in s and int(s[label][0]) != have:
                    notes.append(f"{label.lower()}: benchmark {int(s[label][0])} vs crawl folder {have}")
        parts = [s.get(n, (None,))[0] for n in ("Total wall time (end-to-end)", "Entry discovery time", "Total paper processing time")]
        if None not in parts and abs(parts[0] - parts[1] - parts[2]) > 1.0:
            notes.append("wall time differs from discovery + paper processing by more than 1 s")
        if notes:
            print(f"Check {LABELS[k]}: " + "; ".join(notes))

In [ ]:
def plot_performance(bench, ts):
    keys = [k for k in ("arxiv", "oai") if k in set(bench["_key"])]
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

    # (a) running time (minutes)
    ax = axes[0]
    series = [("Total wall time", "total_wall_time_sec"), ("Entry discovery", "discovery_time_sec"),
              ("Paper processing\n(all papers)", "total_paper_process_time_sec")]
    series = [(n, c) for n, c in series if c in bench.columns]
    x, w = np.arange(len(series)), 0.8 / max(len(keys), 1)
    for i, k in enumerate(keys):
        row = bench[bench["_key"] == k].iloc[0]
        bars = ax.bar(x + (i - (len(keys) - 1) / 2) * w, [float(row[c]) / 60 for _, c in series], w,
                      label=LABELS[k], color=COLORS[k])
        bar_labels(ax, bars, "{:.2f}")
    ax.set_xticks(x); ax.set_xticklabels([n for n, _ in series]); ax.set_ylabel("Minutes")
    ax.set_title("Running time"); ax.legend(frameon=False)

    # (b) RAM over time: record_type == "ram_sample"
    ax = axes[1]
    ram = ts[ts["record_type"] == "ram_sample"] if not ts.empty and "record_type" in ts.columns else pd.DataFrame()
    if not ram.empty:
        for k in keys:
            d = ram[ram["_key"] == k].sort_values("elapsed_sec")
            if d.empty:
                continue
            ax.plot(d["elapsed_sec"] / 60, d["ram_mb"], label=LABELS[k], color=COLORS[k], lw=1.4)
            peak = d.loc[d["ram_mb"].idxmax()]
            ax.scatter(peak["elapsed_sec"] / 60, peak["ram_mb"], s=45, color=COLORS[k], edgecolor="black", zorder=3)
            ax.annotate(f"peak {peak['ram_mb']:.0f} MB", (peak["elapsed_sec"] / 60, peak["ram_mb"]),
                        xytext=(6, 4), textcoords="offset points", fontsize=9)
        ax.set_xlabel("Elapsed time (minutes)"); ax.set_ylabel("Process RAM (MB)"); ax.legend(frameon=False)
    else:
        ax.text(0.5, 0.5, "no RAM samples available", ha="center", va="center", transform=ax.transAxes)
    ax.set_title("RAM footprint over time")

    # (c) storage (MB)
    ax = axes[2]
    items = [("Archives\n(compressed)", "raw_archive_bytes"), ("Before\nfiltering", "total_uncompressed_bytes"),
             ("After\nfiltering", "filtered_uncompressed_bytes"), ("Max disk\nduring crawl", "max_disk_storage_bytes"),
             ("Final\noutput", "final_output_storage_bytes")]
    items = [(n, c) for n, c in items if c in bench.columns]
    x, w = np.arange(len(items)), 0.8 / max(len(keys), 1)
    for i, k in enumerate(keys):
        row = bench[bench["_key"] == k].iloc[0]
        bars = ax.bar(x + (i - (len(keys) - 1) / 2) * w, [float(row[c]) / (1024 ** 2) for _, c in items], w,
                      label=LABELS[k], color=COLORS[k])
        bar_labels(ax, bars, "{:.0f}")
    ax.set_xticks(x); ax.set_xticklabels([n for n, _ in items]); ax.set_ylabel("MB")
    ax.set_title("Disk and storage"); ax.legend(frameon=False)
    return fig

if not bench.empty:
    plot_performance(bench, ts)
    plt.show()

In [ ]:
# Time per paper. The boxplot needs every per-paper row (record_type == "paper_time");
# when the time series holds fewer rows than papers crawled, the summary columns of benchmark_results.csv are drawn instead.
if not bench.empty:
    paper_rows = ts[ts["record_type"] == "paper_time"] if not ts.empty and "record_type" in ts.columns else pd.DataFrame()
    keys = [k for k in ("arxiv", "oai") if k in set(bench["_key"])]
    complete = bool(len(paper_rows)) and all(
        (paper_rows["_key"] == k).sum() >= int(bench.loc[bench["_key"] == k, "papers_crawled"].iloc[0]) for k in keys)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    if complete:
        data = [paper_rows.loc[paper_rows["_key"] == k, "paper_time_sec"].dropna().values for k in keys]
        bp = ax.boxplot(data, tick_labels=[LABELS[k] for k in keys], patch_artist=True, showmeans=True)
        for patch, k in zip(bp["boxes"], keys):
            patch.set_facecolor(COLORS[k]); patch.set_alpha(0.6)
        ax.set_title("Processing time per paper (Semantic Scholar + all versions)")
    else:
        n_have = {LABELS[k]: int((paper_rows["_key"] == k).sum()) if len(paper_rows) else 0 for k in keys}
        print(f"Per-paper rows in the time series: {n_have} (fewer than papers crawled): drawing the summary statistics instead.")
        for i, k in enumerate(keys):
            r = bench[bench["_key"] == k].iloc[0]
            lo, hi, mean, med, sd = (float(r[c]) for c in ("min_paper_time_sec", "max_paper_time_sec", "avg_paper_time_sec",
                                                            "median_paper_time_sec", "std_paper_time_sec"))
            ax.vlines(i, lo, hi, color=COLORS[k], lw=2, alpha=0.5, label="min - max" if i == 0 else None)
            ax.vlines(i, max(lo, mean - sd), min(hi, mean + sd), color=COLORS[k], lw=14, alpha=0.7, label="mean ± std" if i == 0 else None)
            ax.plot(i, med, "o", color="white", mec="black", ms=8, label="median" if i == 0 else None, zorder=3)
            ax.plot(i, mean, "D", color="black", ms=6, label="mean" if i == 0 else None, zorder=3)
        ax.set_xticks(range(len(keys))); ax.set_xticklabels([LABELS[k] for k in keys]); ax.set_xlim(-0.6, len(keys) - 0.4)
        ax.legend(frameon=False)
        ax.set_title("Processing time per paper (summary statistics)")
    ax.set_ylabel("Seconds per paper")
    plt.show()

In [ ]:
# Data-driven reading of the performance table.
lines = []
perf_res = RESULTS.get("perf", {})
wall = {k: next((v for n, v in s.items() if n.startswith("Total wall")), None) for k, s in perf_res.items()}
disc = {k: next((v for n, v in s.items() if n.startswith("Entry discovery")), None) for k, s in perf_res.items()}
if len(wall) == 2 and all(v for v in wall.values()):
    fast, slow = sorted(wall, key=wall.get)
    lines.append(f"- **Total time**: {LABELS[fast]} finished in {wall[fast] / 60:.2f} min and {LABELS[slow]} in {wall[slow] / 60:.2f} min "
                 f"({(1 - wall[fast] / wall[slow]) * 100:.1f}% faster).")
for k, v in disc.items():
    if v is not None and wall.get(k):
        lines.append(f"- **Discovery with {LABELS[k]}**: {v:.1f} s, i.e. {v / wall[k] * 100:.1f}% of its total time.")
for k, s in perf_res.items():
    ram = {n: v for n, v in s.items() if "RAM" in n}
    if ram:
        lines.append(f"- **RAM with {LABELS[k]}**: " + ", ".join(f"{n} {v:.1f} MB" for n, v in ram.items()) + ".")
    fin, raw = s.get("Final output size on disk"), s.get("Size before removing figures")
    if fin and raw:
        lines.append(f"- **Disk with {LABELS[k]}**: final output {fmt_bytes(fin)} vs {fmt_bytes(raw)} before filtering ({(1 - fin / raw) * 100:.1f}% smaller).")
display(Markdown("\n".join(lines) if lines else "_No benchmark data available._"))

# Part IV — Conclusions

## 12. Summary of results

The summary below is generated from the numbers computed in sections 8–11.

In [ ]:
out = []
for k, v in RESULTS.get("validation", {}).items():
    s = f"- **{LABELS[k]}, correctness**: {v['papers']}/{n_expected} papers crawled ({v['success']:.1f}% success rate), {v['versions']} versions"
    s += f"; version count correct for {v['correct']}/{v['covered']} papers ({v['accuracy']:.1f}%) against the Kaggle snapshot." if v["covered"] else "; no ground truth available for version accuracy."
    out.append(s)
for k, st in RESULTS.get("status", {}).items():
    out.append(f"- **{LABELS[k]}, source status**: {st['downloaded']} versions with TeX/Bib, {st['no_tex_or_bib']} without (PDF-only or empty), {st['download_failed']} failed downloads.")
for k, s in RESULTS.get("size", {}).items():
    out.append(f"- **{LABELS[k]}, size**: average {fmt_bytes(s['Average size BEFORE removing figures'])} before vs "
               f"{fmt_bytes(s['Average size AFTER removing figures (.tex + .bib)'])} after removing figures "
               f"({s['Total reduction']:.1f}% total reduction; {s['Average reduction per version']:.1f}% on average per version).")
for k, s in RESULTS.get("refs", {}).items():
    out.append(f"- **{LABELS[k]}, references**: {s['Average references per paper (all papers)']:.2f} per paper on average "
               f"({s['Average references with an arXiv ID per paper']:.2f} with an arXiv ID); API success rate {s['API success rate']:.1f}%, "
               f"but only {s['Share of papers with at least 1 reference']:.1f}% of papers have at least one reference.")
display(Markdown("\n".join(out)))
display(Markdown("""
**Points to keep in mind**

- A reference list of length zero counts as a successful API call; the share of papers with at least one reference is the more informative figure.
- Versions without TeX/Bib are expected (PDF-only or non-TeX submissions) and are kept as empty records, as the milestone requires.
- The arXiv-API crawler stores one `updated_date` for all versions; use the OAI crawler's per-version dates for `revised_dates`.
- Performance depends mainly on the mandatory 3 s pauses; for a known ID list the arXiv API needs far fewer discovery requests, while the OAI crawler is the better fit for date- or set-based harvesting.
"""))